In [1]:
import os
import numpy as np
import pandas as pd



In [2]:
BASE_DIR = "/kaggle/input/siim-isic-melanoma-classification"

train_path = os.path.join(BASE_DIR, "train.csv")
test_path = os.path.join(BASE_DIR, "test.csv")
sample_path = os.path.join(BASE_DIR, "sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sub = pd.read_csv(sample_path)

assert {"image_name", "target"}.issubset(train_df.columns)
assert {"image_name"}.issubset(test_df.columns)
assert list(sub.columns) == ["image_name", "target"]



In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_sample_weight

train_df = train_df.copy()
test_df = test_df.copy()


# BUGFIX: ensure categorical cleaning produces numpy-friendly np.nan (not pd.NA),
# because sklearn's SimpleImputer/_get_mask can error on pd.NA with object arrays.
def _clean_cat(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = s.str.strip()
    s = s.replace("", pd.NA)
    s = s.str.lower()
    # convert to object with np.nan for missing
    s = s.astype(object).where(~pd.isna(s), np.nan)
    return s


for _df in (train_df, test_df):
    if "sex" in _df.columns:
        _df["sex"] = _clean_cat(_df["sex"])
    if "anatom_site_general_challenge" in _df.columns:
        _df["anatom_site_general_challenge"] = _clean_cat(
            _df["anatom_site_general_challenge"]
        )

for _df in (train_df, test_df):
    if "age_approx" in _df.columns:
        _df["age_approx"] = pd.to_numeric(_df["age_approx"], errors="coerce")

features = [
    "sex",
    "age_approx",
    "anatom_site_general_challenge",
    "patient_id",
]
target_col = "target"

X = train_df[features].copy()
y = train_df[target_col].astype(int).values
X_test = test_df[features].copy()


class AddMissingIndicator(BaseEstimator, TransformerMixin):
    def __init__(self, col_name, out_name=None):
        self.col_name = col_name
        self.out_name = out_name or (col_name + "_isna")

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # BUGFIX: use pandas isna -> float32; avoid any pd.NA boolean ambiguity
        s = (
            pd.Series(X[self.col_name])
            .isna()
            .astype(np.float32)
            .to_numpy()
            .reshape(-1, 1)
        )
        return s


class AgeSquared(BaseEstimator, TransformerMixin):
    def __init__(self, col_name="age_approx"):
        self.col_name = col_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # BUGFIX: ensure output is numeric ndarray with np.nan (not pd.NA)
        s = pd.to_numeric(pd.Series(X[self.col_name]), errors="coerce").to_numpy(
            dtype=np.float32
        )
        s2 = (s**2).astype(np.float32).reshape(-1, 1)
        return s2


class SexAgeInteraction(BaseEstimator, TransformerMixin):
    def __init__(self, sex_col="sex", age_col="age_approx"):
        self.sex_col = sex_col
        self.age_col = age_col
        self.age_median_ = None
        self.age_mean_ = None
        self.age_std_ = None

    def fit(self, X, y=None):
        age = pd.to_numeric(pd.Series(X[self.age_col]), errors="coerce")
        self.age_median_ = float(age.median())
        age_filled = age.fillna(self.age_median_)
        self.age_mean_ = float(age_filled.mean())
        self.age_std_ = float(age_filled.std(ddof=0))
        if not np.isfinite(self.age_std_) or self.age_std_ == 0.0:
            self.age_std_ = 1.0
        return self

    def transform(self, X):
        # BUGFIX: do not create string "NA" category; keep missing as "missing" sentinel
        sex = pd.Series(X[self.sex_col])
        sex = sex.astype(object).where(~pd.isna(sex), "missing")
        sex = sex.astype(str).str.lower()
        male = sex.isin(["male", "m"]).astype(np.float32).to_numpy().reshape(-1, 1)

        age = pd.to_numeric(pd.Series(X[self.age_col]), errors="coerce")
        age = age.fillna(self.age_median_).to_numpy(dtype=np.float32).reshape(-1, 1)
        age_z = (age - self.age_mean_) / self.age_std_
        return (male * age_z).astype(np.float32)


cat_cols = ["sex", "anatom_site_general_challenge"]
num_cols = ["age_approx"]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            num_cols,
        ),
        (
            "age2",
            Pipeline(
                steps=[
                    ("age2", AgeSquared(col_name="age_approx")),
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            ["age_approx", "patient_id", "sex", "anatom_site_general_challenge"],
        ),
        (
            "sex_age_int",
            SexAgeInteraction(sex_col="sex", age_col="age_approx"),
            ["age_approx", "patient_id", "sex", "anatom_site_general_challenge"],
        ),
        (
            "age_isna",
            AddMissingIndicator(col_name="age_approx", out_name="age_approx_isna"),
            ["age_approx", "patient_id", "sex", "anatom_site_general_challenge"],
        ),
        (
            "sex_isna",
            AddMissingIndicator(col_name="sex", out_name="sex_isna"),
            ["age_approx", "patient_id", "sex", "anatom_site_general_challenge"],
        ),
        (
            "site_isna",
            AddMissingIndicator(
                col_name="anatom_site_general_challenge",
                out_name="anatom_site_general_challenge_isna",
            ),
            ["age_approx", "patient_id", "sex", "anatom_site_general_challenge"],
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
                ]
            ),
            cat_cols,
        ),
    ],
    remainder="drop",
)

clf = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=2.0,
    max_iter=4000,
    n_jobs=None,
    class_weight=None,
    random_state=42,
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", clf),
    ]
)



In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

patient_ids = train_df["patient_id"].astype(str)
patient_level = (
    pd.DataFrame({"patient_id": patient_ids, "target": y})
    .groupby("patient_id", as_index=False)["target"]
    .max()
)
patient_arr = patient_level["patient_id"].values
patient_y = patient_level["target"].astype(int).values

test_pred = np.zeros(len(test_df), dtype=np.float64)

for fold, (ptr_idx, pva_idx) in enumerate(skf.split(patient_arr, patient_y), 1):
    tr_patients = set(patient_arr[ptr_idx])
    va_patients = set(patient_arr[pva_idx])

    tr_mask = patient_ids.isin(tr_patients).values
    assert not np.any(tr_mask & patient_ids.isin(va_patients).values)

    X_tr, y_tr = X.loc[tr_mask], y[tr_mask]

    sw = compute_sample_weight(class_weight="balanced", y=y_tr)

    model.fit(X_tr, y_tr, clf__sample_weight=sw)

    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits

test_pred = np.clip(test_pred, 0.0, 1.0)



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [5]:
pred_map = pd.DataFrame(
    {"image_name": test_df["image_name"].values, "target": test_pred}
)
sub = sub.merge(pred_map, on="image_name", how="left", suffixes=("_old", ""))

if sub["target"].isna().any():
    sub["target"] = sub["target"].fillna(float(np.mean(test_pred)))

sub = sub[["image_name", "target"]]
sub.to_csv("submission.csv", index=False)

sub.head()



,image_name,target
0,ISIC_0052212,0.453333
1,ISIC_0076545,0.554667
2,ISIC_0085172,0.453333
3,ISIC_0086709,0.265041
4,ISIC_0109568,0.759363


In [6]:
assert os.path.exists("submission.csv")
chk = pd.read_csv("submission.csv")
assert list(chk.columns) == ["image_name", "target"]
assert len(chk) == len(pd.read_csv(sample_path))
chk.describe(include="all")

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,image_name,target
count,4142,4142.000000
unique,4142,NaN
top,ISIC_9998965,NaN
freq,1,NaN
mean,NaN,0.458602
std,NaN,0.175521
min,NaN,0.068515
25%,NaN,0.305961
50%,NaN,0.419597
75%,NaN,0.558997
